# API Integration — Pagination, Retries & Backoff, Rate Limiting

**DevRev Technical Round · Section 1.** A concise, *runnable* walkthrough of the three concepts that always show up together when a client pulls data from a third-party API:

1. **Pagination** — the data doesn't fit in one response.
2. **Retries & backoff** — the network/server is unreliable.
3. **Rate limiting** — the server throttles you if you go too fast.

**Mental model:** pagination is the *outer loop*; rate-limiting *gates* each call; retry *wraps* each call.

```mermaid
flowchart LR
    A["fetch_all_pages()"] --> B["rate limiter:\nwait for a token"]
    B --> C["retry wrapper:\nbackoff on failure"]
    C --> D["HTTP request"]
    D -->|"200 + next_cursor"| A
    D -->|"429 / 5xx"| C
```

Each section below: **the concept** → **the code** → **a live demo that proves the point** (especially the failure modes — that's what interviewers actually probe).

```mermaid
flowchart LR
    A["fetch_all_pages()"] --> B["rate limiter:\nwait for a token"]
    B --> C["retry wrapper:\nbackoff on failure"]
    C --> D["HTTP request"]
    D -->|"200 + next_cursor"| A
    D -->|"429 / 5xx"| C
```

---
## 1. Pagination

**Cursor** pagination: each response hands you an opaque token for the next page (`GET ?cursor=abc123`). **Offset** pagination: you compute the next window yourself by skipping N rows (`GET ?offset=200`).

| | Offset | Cursor |
|---|---|---|
| Random access ("jump to page 50") | ✅ easy | ❌ must walk sequentially |
| Stable under concurrent inserts/deletes | ❌ **no** | ✅ yes |
| Cost on deep pages | ❌ `OFFSET 1000000` scans | ✅ cheap indexed seek |

**Interview one-liner:** *"Offset is simpler and allows random access, but silently skips or duplicates rows when data changes mid-scan. Cursor is stable and cheap on deep pages — what production APIs standardize on, at the cost of no random access."*

In [2]:
def fetch_all_pages(fetch_page, start_cursor=None, max_pages=10_000):
    """Loop until there is no next cursor.
    fetch_page(cursor) -> {"items": [...], "next_cursor": "abc" or None}
    """
    results, cursor, pages = [], start_cursor, 0
    while True:
        page = fetch_page(cursor)
        results.extend(page["items"])
        cursor = page.get("next_cursor")       # None / "" / missing key all mean "stop"
        pages += 1
        if not cursor:
            break
        if pages >= max_pages:                 # safety cap against a cursor loop / server bug
            raise RuntimeError("pagination exceeded max_pages")
    return results


records = list(range(1, 24))  # pretend server-side dataset, 23 rows

def cursor_page(cursor):
    start = cursor or 0
    chunk = records[start:start + 10]
    result = {"items": chunk, "next_cursor": start + 10 if start + 10 < len(records) else None}
    print(f"result {result}")
    return result

print("cursor pagination collected:", fetch_all_pages(cursor_page))

result {'items': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 'next_cursor': 10}
result {'items': [11, 12, 13, 14, 15, 16, 17, 18, 19, 20], 'next_cursor': 20}
result {'items': [21, 22, 23], 'next_cursor': None}
cursor pagination collected: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


### The failure mode interviewers actually probe: offset pagination + a concurrent delete

Below, the server has rows `[A, B, C, D, E]`. We page with `offset=0,limit=2`, then — *before* the next request — someone deletes `A`. Watch what happens to `C`.

In [3]:
server_rows = ["A", "B", "C", "D", "E"]

page1 = server_rows[0:2]
print("page 1 (offset=0, limit=2):", page1)

server_rows.remove("A")  # concurrent delete happens between our two requests
print("someone deletes 'A' -> server rows are now:", server_rows)

page2 = server_rows[2:4]  # client still asks for offset=2 ("skip the 2 rows I already saw")
print("page 2 (offset=2, limit=2):", page2)
print("'C' was never returned ->", "C" not in page1 and "C" not in page2)

page 1 (offset=0, limit=2): ['A', 'B']
someone deletes 'A' -> server rows are now: ['B', 'C', 'D', 'E']
page 2 (offset=2, limit=2): ['D', 'E']
'C' was never returned -> True


`C` fell through the gap: offset means *"skip N rows"*, and after the delete, `offset=2` now points past `C`. A concurrent **insert** before the cursor has the opposite effect — you'd see a row **twice**. Cursor pagination avoids both because the token means *"give me rows after this specific row"*, which doesn't shift when unrelated rows are added/removed elsewhere.

---
## 2. Retries & Backoff

**Retry** transient failures: `429` (throttled), `5xx` (server hiccup), network timeouts.
**Never retry** client errors (`400/401/403/404/422`) — they'll fail identically every time.

**Exponential backoff:** wait `base × 2^attempt` (capped), so a struggling server gets room to recover.
**Full jitter:** sleep a *random* duration in `[0, ceiling]` instead of exactly `ceiling` — otherwise every client that failed at the same instant retries at the *same* future instant ("thundering herd"). Demonstrated below.

In [ ]:
import random

RETRYABLE_STATUS = {429, 500, 502, 503, 504}

class HTTPError(Exception):
    def __init__(self, status):
        super().__init__(f"HTTP {status}")
        self.status = status

def retry_with_backoff(max_retries=5, base=0.5, cap=30.0, sleep=lambda d: None):
    """Decorator: retry transient HTTP errors with exponential backoff + full jitter."""
    def decorator(fn):
        def wrapper(*args, **kwargs):
            attempt = 0
            while True:
                try:
                    return fn(*args, **kwargs)
                except HTTPError as e:
                    if e.status not in RETRYABLE_STATUS or attempt >= max_retries:
                        raise                              # fatal, or out of attempts
                    ceiling = min(cap, base * (2 ** attempt))
                    sleep(random.uniform(0, ceiling))       # FULL JITTER, not `ceiling` itself
                    attempt += 1
        return wrapper
    return decorator

calls = {"n": 0}
@retry_with_backoff(base=0.01, cap=0.05)  # sleep is a no-op here, we're demoing the logic not the wait
def flaky_call():
    calls["n"] += 1
    if calls["n"] < 3:
        raise HTTPError(429)   # throttled twice, then succeeds
    return "ok"

print("result:", flaky_call(), "after", calls["n"], "attempts")

try:
    retry_with_backoff()(lambda: (_ for _ in ()).throw(HTTPError(404)))()
except HTTPError as e:
    print(f"HTTP {e.status} raised immediately -- not retried (client error)")

### Why jitter matters: 6 workers that all get throttled at the same instant

In [ ]:
ceiling = 1.0  # base * 2^attempt for their shared attempt number

no_jitter = [ceiling] * 6                              # everyone waits the SAME fixed delay
with_jitter = [round(random.uniform(0, ceiling), 2) for _ in range(6)]  # full jitter

print("no jitter  -> all retry at:", no_jitter, " (synchronized stampede back onto the server)")
print("full jitter-> retry spread:", sorted(with_jitter), " (load smoothed over the window)")

---
## 3. Rate Limiting — Token Bucket

A bucket holds up to `capacity` tokens, refilled at `rate` tokens/sec. Each request spends one token; an empty bucket means *wait*. This allows a **burst** (up to `capacity`) while enforcing the **average** `rate` over time — which is how most rate-limited REST APIs actually advertise their limits ("300 req/min with bursting").

**Token bucket vs leaky bucket:** token bucket lets you spend saved-up allowance as a burst; leaky bucket forces a perfectly even output rate (use it only when the downstream truly can't handle any spike).

In [ ]:
import time, threading

class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate, self.capacity = rate, capacity
        self.tokens = float(capacity)           # start full -> allows an immediate burst
        self.last = time.monotonic()
        self.lock = threading.Lock()            # makes "check + spend" atomic across threads

    def _refill(self):
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now

    def acquire(self, n=1):
        while True:
            with self.lock:
                self._refill()
                if self.tokens >= n:
                    self.tokens -= n
                    return
                wait = (n - self.tokens) / self.rate
            time.sleep(wait)                    # sleep OUTSIDE the lock so others aren't blocked

# rate=20/s, capacity=5: expect 5 requests instantly (the burst), then throttled to ~20/s.
bucket = TokenBucket(rate=20, capacity=5)
start = time.monotonic()
for _ in range(15):
    bucket.acquire()
elapsed = time.monotonic() - start
print(f"15 requests through a 20/s bucket (burst 5) took {elapsed:.2f}s (~0.5s expected: 10 extra / 20 per sec)")

---
## Cheat Sheet

| Topic | 15-second answer |
|---|---|
| **Pagination** | Cursor over offset: stable under concurrent writes, cheap on deep pages; loop until `next_cursor` is null, with a max-page safety cap. |
| **Offset drift** | Deletes before the cursor skip rows; inserts duplicate them — offset means "skip N", not "after row X". |
| **Retries** | Retry only transient errors (429, 5xx, timeouts); exponential backoff with **full jitter**, a per-attempt cap, and a total time budget. |
| **Jitter** | Without it, all failed clients retry at the same instant → thundering herd. Random delay spreads the load. |
| **Token bucket** | Tokens refill at `rate`, capped at `capacity` (= max burst). Spend one per request; refill lazily on each call — no background thread needed. |
| **Token vs leaky** | Token bucket allows bursts (matches most REST APIs); leaky bucket forces perfectly even output (fragile downstream). |
| **Shared limit** | One locked bucket for threads in a process; a Redis-backed atomic bucket across processes/machines. |

Full production-grade version (with idempotent webhook handling + a composed `RateLimitedClient`) lives in [`api_integration_reference.ipynb`](api_integration_reference.ipynb); deeper narrative + diagrams in [`01_API_Integration.md`](01_API_Integration.md).